# Generative Adversarial Networks — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## the minimax generator-discriminator game
Binary classification supplies the discriminator loss, neural-network optimization supplies alternating gradient steps, and distribution matching explains why fooling the discriminator can imply realistic samples. WGANs and conditional GANs (10.6), StyleGAN (10.7), and GAN evaluation (10.8) all modify this game.

In [ ]:
# 1) discriminator separates real and fake 1D samples
x=np.linspace(-3,4,200); real=normal_pdf(x,0,0.7); fake=normal_pdf(x,1.5,0.8); D=real/(real+fake)
plt.figure(figsize=(5,3)); plt.plot(x,D); plt.ylim(0,1); plt.title('D(x) high where real density dominates')
assert D[np.argmin(abs(x))]>D[np.argmin(abs(x-1.5))]

In [ ]:
# 2) generator mean shift changes discriminator score
means=np.linspace(2,-.2,60); score=[]
for m in means:
    fake=normal_pdf(x,m,0.8); DD=real/(real+fake); score.append(np.mean(DD))
plt.figure(figsize=(5,3)); plt.plot(means,score); plt.xlabel('fake mean'); plt.title('moving fake toward real fools D')
assert score[-1]<score[0]

In [ ]:
# 3) minimax terms for one pair of probabilities
Dr=0.73; Df=0.41; vals=[math.log(Dr), math.log(1-Df), math.log(Dr)+math.log(1-Df)]
plt.figure(figsize=(4,3)); plt.bar(['real','fake','sum'],vals); plt.title('GAN discriminator objective pieces')
assert abs(vals[2]+0.842)<0.01

In [ ]:
# 4) mode collapse can match one peak and miss another
real_mix=0.5*normal_pdf(x,-1,.35)+0.5*normal_pdf(x,1,.35); collapsed=normal_pdf(x,1,.35)
plt.figure(figsize=(5,3)); plt.plot(x,real_mix,label='real two modes'); plt.plot(x,collapsed,label='collapsed'); plt.legend(); plt.title('one mode is missing')
assert collapsed[np.argmin(abs(x+1))]<real_mix[np.argmin(abs(x+1))]

In [ ]:
# 5) alternating toy updates move generator toward real mean
m=2.0; path=[]
for _ in range(20):
    grad=m-0.0; m-=0.15*grad; path.append(m)
plt.figure(figsize=(5,3)); plt.plot(path,'o-'); plt.title('generator parameter approaches data mean')
assert abs(path[-1])<abs(path[0])